# ACE-Step 1.5 score candidates for Heritage Short (Kaggle / Colab GPU)

Generates **instrumental score candidates with the real ACE-Step 1.5 model** (MIT licence, DiT-only so it fits a 16 GB T4). This replaces the earlier version of this notebook, which loaded `facebook/musicgen-small` (CC-BY-NC) and fell back to a synthetic pad while labelling everything "ACE-Step". If generation fails here, the candidate is logged as an **error**; there is no silent fallback.

### Workflow
1. **Draft pass** (`MODE = "draft"`, turbo model, 8 steps, about a minute or two per candidate): audition caption variants **A / B / C** and pick the one that sounds closest to Ethiopian instruments rather than generic world music.
2. **Final pass** (`MODE = "final"`, SFT model, 32 steps): set `VARIANTS_TO_RUN` to your favourite and generate a few seeds with the higher-quality model.
3. Audition the previews, download the zip, copy your favourite into the repo and lock it in `storyboard.json` (the last cell prints exactly what to change).

### Requirements
Accelerator: **GPU (T4 is enough)** and internet on. Upload `storyboard.json` next to this notebook (optional: it supplies bpm, key, duration, seed, and caption A). The first run downloads Python 3.11/3.12 via `uv`, a few GB of dependencies, and model weights, so expect 10-20 minutes before the first candidate.

### Honest caveat
The runner (`notebooks/acestep_score.py`, embedded below) was written against ACE-Step 1.5's documented API and checked against its source at commit `ca1e85f`; its control flow was tested with a stand-in model. It has **not** been run on a real GPU by its author. If the model does not render masenqo or krar convincingly, that is a limit of the model's training data, and the fix is a human-recorded track (see the repo README).

In [ ]:
# Cell 1: Settings  --  the only cell you normally edit
MODE = "draft"                 # "draft": turbo, 8 steps (audition PROMPTS)   |   "final": SFT, 32 steps (best quality)
VARIANTS_TO_RUN = ["A", "B", "C"]   # in "final" mode narrow this to your favourite, e.g. ["A"]
SEEDS_PER_VARIANT = 2          # seeds = base_seed, base_seed+1, ...  (base_seed comes from storyboard.json)
PREVIEW_SECONDS = 45           # length of the inline audition clips (full-length WAVs are still saved)

MODELS = {"draft": ("acestep-v15-turbo", 8), "final": ("acestep-v15-sft", 32)}
MODEL, STEPS = MODELS[MODE]

ACESTEP_REPO = "https://github.com/ACE-Step/ACE-Step-1.5.git"
ACESTEP_COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"          # version this notebook was written against (falls back to latest if unavailable)
RUN_CMD = "uv run python"              # runs the script inside ACE-Step's own pinned environment

CAPTIONS = {
    "A": "Traditional Ethiopian instrumental in the Tizita mode, the mode of memory and longing: a solo masenqo, a single-string bowed fiddle, sings the melody with expressive slides and ornaments over gently plucked krar lyre arpeggios and a soft kebero hand-drum heartbeat. Sparse, slow, intimate and nostalgic, warm live acoustic recording in a small room, no vocals, no synthesizers, no drum kit.",
    "B": "Slow meditative Ethiopian instrumental: a krar lyre plays a hypnotic pentatonic ostinato with plenty of silence between the notes, a warm solo cello pad swells underneath, a soft washint bamboo flute answers in the gaps, faint rain on a tin roof in the background. Intimate and nostalgic, dry close-miked acoustic recording, no vocals, no synthesizers, no drums.",
    "C": "Slow 1970s Addis Ababa Ethio-jazz instrumental: warm vibraphone melody in a pentatonic minor mode, muted upright bass, brushed drums at a gentle pulse, soft Rhodes chords, a wistful bowed fiddle fill now and then. Rainy-afternoon nostalgia, analog tape warmth, spacious and unhurried, no vocals.",
}
print(f"MODE={MODE}  model={MODEL}  steps={STEPS}  variants={VARIANTS_TO_RUN}  seeds/variant={SEEDS_PER_VARIANT}")


In [ ]:
# Cell 2: Environment check + install ACE-Step 1.5 in an isolated uv environment
import os, subprocess, sys, shutil

WORK = os.getcwd()
REPO_DIR = os.path.join(WORK, "ACE-Step-1.5")

# Colab/Kaggle export MPLBACKEND=module://matplotlib_inline..., which ACE-Step's own venv does not
# contain -> matplotlib import crash. Force a headless backend for every subprocess.
SUB_ENV = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1")

def sh(cmd, check=True):
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=SUB_ENV)
    for line in p.stdout:
        print("  " + line.rstrip())
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

print("python", sys.version.split()[0])
if sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False) != 0:
    raise SystemExit("No GPU detected. In Kaggle: Settings > Accelerator > GPU T4. In Colab: Runtime > Change runtime type > GPU.")

sh("pip install -q uv")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    rc = sh(f"git init -q {REPO_DIR} && cd {REPO_DIR} && git remote add origin {ACESTEP_REPO} "
            f"&& git fetch -q --depth 1 origin {ACESTEP_COMMIT} && git checkout -q FETCH_HEAD", check=False)
    if rc != 0:
        print("[warn] pinned commit unavailable; using latest ACE-Step main instead")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        sh(f"git clone -q --depth 1 {ACESTEP_REPO} {REPO_DIR}")
sh(f"cd {REPO_DIR} && uv sync")     # creates ACE-Step's own .venv (Python 3.11-3.12) with pinned torch/transformers
print("\n[+] ACE-Step environment ready.")


In [ ]:
# Cell 3: Build config.json from storyboard.json (if uploaded) and write the runner script
import json, os

OUT_DIR = os.path.join(WORK, "outputs", "music")
os.makedirs(OUT_DIR, exist_ok=True)
RUNS_LOG = os.path.join(OUT_DIR, "music_runs.jsonl")

music = {}
if os.path.exists("storyboard.json"):
    with open("storyboard.json", "r", encoding="utf-8") as f:
        music = json.load(f).get("project", {}).get("music", {})
    print("Loaded project.music from storyboard.json")
else:
    print("storyboard.json not found; using built-in defaults (upload it to inherit your settings)")

bpm = int(music.get("bpm", 68))
keyscale = music.get("key", "D minor")
duration_s = float(music.get("duration_s", 82.0))
base_seed = int(music.get("seed", 1974))
captions = dict(CAPTIONS)
if music.get("caption"):
    captions["A"] = music["caption"]          # variant A mirrors the storyboard so the two never drift apart

config = {
    "project_root": REPO_DIR, "out_dir": OUT_DIR, "runs_log": RUNS_LOG,
    "model": MODEL, "steps": STEPS, "mode": MODE,
    "bpm": bpm, "keyscale": keyscale, "duration_s": duration_s,
    "quantization": "auto", "offload_to_cpu": "auto",
    "seeds": [base_seed + i for i in range(SEEDS_PER_VARIANT)],
    "variants": [{"id": v, "caption": captions[v]} for v in VARIANTS_TO_RUN],
}
CONFIG_PATH = os.path.join(WORK, "acestep_config.json")
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

RUNNER_SRC = r'''"""ACE-Step 1.5 score-candidate generator for heritage-short.

Runs on a CUDA GPU (Kaggle T4, Colab, or local) INSIDE the ACE-Step checkout's own
environment, because ACE-Step pins its own torch / transformers versions:

    cd ACE-Step-1.5 && uv run python /path/to/acestep_score.py /path/to/config.json

It is driven by a JSON config (written by notebooks/acestep_score.ipynb) and, for every
(caption variant x seed), generates one instrumental candidate, writes
`<out_dir>/score_cand_<variant>_<seed>.wav`, and appends one record per candidate to
`<runs_log>` following the runs.jsonl contract (see ARCHITECTURE.md section 5), labelled with
the model that REALLY ran. If generation fails, the candidate is logged as an error. There is
no silent synthetic fallback, so a bad run cannot be mistaken for a model result.

API used (verified against ACE-Step-1.5 @ ca1e85f, MIT licence):
    AceStepHandler().initialize_service(project_root, config_path, device, offload_to_cpu, quantization)
    generate_music(dit_handler, llm_handler=None, GenerationParams, GenerationConfig, save_dir)
The 5Hz language model is deliberately skipped (thinking=False, llm_handler=None): captions here
are fully specified, and DiT-only mode fits a 16 GB T4 comfortably.
"""

from __future__ import annotations

import json
import os
import shutil
import sys
import tempfile
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

MIN_PEAK = 0.02  # below this the output is treated as near-silent (a known failure mode)


def detect_platform() -> str:
    """Where this GPU time was spent (stored in params.platform so the runs.jsonl schema is unchanged)."""
    env = os.environ
    if env.get("COLAB_RELEASE_TAG") or env.get("COLAB_GPU"):
        return "colab"
    if env.get("KAGGLE_KERNEL_RUN_TYPE") or env.get("KAGGLE_URL_BASE"):
        return "kaggle"
    return "local"


def log_record(path: str, **fields: Any) -> None:
    rec: Dict[str, Any] = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "stage": "music",
        "shot_id": None,
        "model": fields.pop("model"),
        "params": dict(fields.pop("params"), platform=detect_platform()),
        "seed": fields.pop("seed"),
        "latency_s": round(float(fields.pop("latency_s")), 4),
        "status": fields.pop("status"),
        "gpu_min": round(float(fields.pop("gpu_min")), 4),
        "est_cost_usd": 0.0,
        "cache_hit": False,
        "error": fields.pop("error", None),
    }
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False, separators=(",", ":")) + "\n")


def gpu_vram_gb() -> Optional[float]:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory / 1024**3
    except Exception:
        pass
    return None


def init_dit(dit: Any, cfg: Dict[str, Any], vram_gb: float) -> Tuple[Optional[str], bool]:
    """Initialise the DiT, stepping down to safer memory settings if a try fails."""
    model = cfg["model"]
    root = cfg["project_root"]
    small = vram_gb < 20.0  # ACE-Step docs: INT8 + CPU offload are the defaults below ~20 GB

    quant_cfg = cfg.get("quantization", "auto")
    quant = ("int8_weight_only" if small else None) if quant_cfg == "auto" else (quant_cfg or None)
    off_cfg = cfg.get("offload_to_cpu", "auto")
    offload = small if off_cfg == "auto" else bool(off_cfg)

    # Pre-Ampere GPUs (T4/P100, compute capability < 8) make ACE-Step fall back to float16, which
    # overflows to NaN latents ("Generation produced NaN or Inf latents"). bfloat16 has float32's
    # range, so force it (emulated on Turing: slower but numerically safe). The ACESTEP_DTYPE env var
    # suggested by the error message is NOT read by this ACE-Step version. Opt out: "dtype_mode": "fp16".
    try:
        import torch
        from acestep import gpu_config
        major = torch.cuda.get_device_capability()[0]
        if major < 8 and cfg.get("dtype_mode", "bf16") == "bf16":
            gpu_config.cuda_supports_bfloat16 = lambda *a, **k: True
            print(f"[dtype] compute capability {major}.x < 8.0: forcing bfloat16 instead of float16 (avoids NaN latents)")
    except Exception as exc:  # noqa: BLE001
        print(f"[dtype] could not apply bfloat16 override: {exc}")

    attempts: List[Tuple[Optional[str], bool]] = []
    for a in ((quant, offload), (None, offload), (None, True)):
        if a not in attempts:
            attempts.append(a)

    for q, off in attempts:
        try:
            res = dit.initialize_service(
                project_root=root, config_path=model, device="cuda", offload_to_cpu=off, quantization=q
            )
            ok = res[1] if isinstance(res, tuple) else bool(res)
            msg = res[0] if isinstance(res, tuple) else ""
        except Exception as exc:  # noqa: BLE001
            ok, msg = False, f"{type(exc).__name__}: {exc}"
        print(f"[init] {model} quantization={q} offload_to_cpu={off} -> {'OK' if ok else 'FAILED'} {str(msg)[:200]}")
        if ok:
            return q, off
    raise RuntimeError(f"Could not initialise {model} with any memory configuration")


def peak_and_seconds(audio: Dict[str, Any]) -> Tuple[Optional[float], Optional[float]]:
    tensor = audio.get("tensor")
    sr = audio.get("sample_rate") or 48000
    try:
        peak = float(abs(tensor).max())
        seconds = float(tensor.shape[-1]) / float(sr)
        return peak, seconds
    except Exception:
        return None, None


def main(cfg_path: str) -> int:
    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    vram = gpu_vram_gb()
    if vram is None:
        print("[FATAL] No CUDA GPU detected. In Kaggle choose Accelerator = GPU T4; in Colab Runtime > GPU.", file=sys.stderr)
        return 2
    print(f"[env] GPU VRAM {vram:.1f} GB | model {cfg['model']} | steps {cfg['steps']} | mode {cfg.get('mode')}")

    from acestep.handler import AceStepHandler
    from acestep.inference import GenerationConfig, GenerationParams, generate_music

    dit = AceStepHandler()
    quant, offload = init_dit(dit, cfg, vram)

    out_dir = cfg["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    model_label = f"ACE-Step-1.5/{cfg['model']}"
    ok_count = fail_count = 0

    for variant in cfg["variants"]:
        for seed in cfg["seeds"]:
            name = f"score_cand_{variant['id']}_{seed}.wav"
            dest = os.path.join(out_dir, name)
            params_log = {
                "variant": variant["id"],
                "caption": variant["caption"],
                "bpm": cfg["bpm"],
                "keyscale": cfg["keyscale"],
                "duration_s": cfg["duration_s"],
                "steps": cfg["steps"],
                "quantization": quant,
                "offload_to_cpu": offload,
                "instrumental": True,
                "output_file": name,
            }
            print(f"[gen] variant {variant['id']} seed {seed} ...")
            t0 = time.time()
            tmp = tempfile.mkdtemp(prefix="acestep_")
            try:
                params = GenerationParams(
                    task_type="text2music",
                    caption=variant["caption"],
                    lyrics="[Instrumental]",
                    instrumental=True,
                    bpm=int(cfg["bpm"]),
                    keyscale=cfg["keyscale"],
                    vocal_language="unknown",
                    duration=float(cfg["duration_s"]),
                    inference_steps=int(cfg["steps"]),
                    seed=int(seed),
                    thinking=False,
                    use_cot_metas=False,
                    use_cot_caption=False,
                    use_cot_language=False,
                )
                gen_cfg = GenerationConfig(
                    batch_size=1, use_random_seed=False, seeds=[int(seed)], audio_format="wav"
                )
                result = generate_music(dit, None, params, gen_cfg, save_dir=tmp)
                if not result.success or not result.audios:
                    raise RuntimeError(f"generate_music failed: {result.error or result.status_message}")
                audio = result.audios[0]
                src = audio.get("path")
                if not src or not os.path.exists(src):
                    raise RuntimeError("generate_music reported success but wrote no audio file")
                peak, seconds = peak_and_seconds(audio)
                if peak is not None and peak < MIN_PEAK:
                    raise RuntimeError(f"near-silent output (peak {peak:.4f} < {MIN_PEAK})")
                shutil.copyfile(src, dest)
                latency = time.time() - t0
                params_log.update(peak=None if peak is None else round(peak, 4),
                                  actual_duration_s=None if seconds is None else round(seconds, 2),
                                  sample_rate=audio.get("sample_rate"))
                log_record(cfg["runs_log"], model=model_label, params=params_log, seed=int(seed),
                           latency_s=latency, status="ok", gpu_min=latency / 60.0)
                ok_count += 1
                print(f"  [+] {dest} ({seconds and round(seconds, 1)}s audio, {latency:.0f}s, peak {peak})")
            except Exception as exc:  # noqa: BLE001
                latency = time.time() - t0
                log_record(cfg["runs_log"], model=model_label, params=params_log, seed=int(seed),
                           latency_s=latency, status="error", gpu_min=latency / 60.0, error=f"{type(exc).__name__}: {exc}")
                fail_count += 1
                print(f"  [-] {name}: {exc}", file=sys.stderr)
            finally:
                shutil.rmtree(tmp, ignore_errors=True)
                try:
                    import torch

                    torch.cuda.empty_cache()
                except Exception:
                    pass

    print(f"\n[done] {ok_count} candidate(s) ok, {fail_count} failed -> {out_dir}")
    return 0 if ok_count else 1


if __name__ == "__main__":
    if len(sys.argv) != 2:
        print("usage: acestep_score.py <config.json>", file=sys.stderr)
        sys.exit(2)
    sys.exit(main(sys.argv[1]))
'''
RUNNER_PATH = os.path.join(WORK, "acestep_score.py")
with open(RUNNER_PATH, "w", encoding="utf-8") as f:
    f.write(RUNNER_SRC)

print(f"{len(config['variants'])} variant(s) x {len(config['seeds'])} seed(s) = "
      f"{len(config['variants']) * len(config['seeds'])} candidate(s); {duration_s:.0f}s each, {bpm} BPM, {keyscale}")


In [ ]:
# Cell 4: Generate candidates (the slow cell: weights download on first run)
rc = sh(f"cd {REPO_DIR} && {RUN_CMD} {RUNNER_PATH} {CONFIG_PATH}", check=False)
print("\nrunner exit code:", rc, "(0 = at least one candidate succeeded)")


In [ ]:
# Cell 5: Audition. Short previews are embedded so the notebook stays small; full WAVs are in outputs/music/
import glob, shutil, subprocess
from IPython.display import Audio, display

previews = sorted(glob.glob(os.path.join(OUT_DIR, "score_cand_*.wav")))
if not previews:
    print("No candidates were produced. Read the errors above and in", RUNS_LOG)
have_ffmpeg = shutil.which("ffmpeg") is not None
for p in previews:
    print(os.path.basename(p))
    if have_ffmpeg:
        mp3 = p.replace(".wav", "_preview.mp3")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", p, "-t", str(PREVIEW_SECONDS),
                        "-b:a", "96k", mp3], check=True)
        display(Audio(mp3))
    else:
        display(Audio(p))


In [ ]:
# Cell 6: Package for download and print how to lock in your favourite
import zipfile

zip_filename = "music_candidates_bundle.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for name in files:
            if name.endswith("_preview.mp3"):
                continue
            fp = os.path.join(root, name)
            zf.write(fp, arcname=os.path.relpath(fp, os.path.join(WORK, "outputs")))
print(f"Bundle ready for download: {zip_filename}\n")
print("""To lock in a candidate (say variant A, seed 1974):
  1. Extract the zip into the repo's outputs/ folder and copy
       outputs/music/score_cand_A_1974.wav  ->  outputs/music/heritage_theme.wav
  2. In storyboard.json set  project.music.selected_file = "outputs/music/heritage_theme.wav"
     and project.music.caption to the caption of the variant you chose (variant A already matches).
  3. Run:  make music assemble        (music.py merges music_runs.jsonl into runs.jsonl automatically)
Not happy? Re-run with MODE = "final" and a different VARIANTS_TO_RUN / SEEDS_PER_VARIANT.""")
